In [1]:
import os
import sys
from dotenv import load_dotenv

# Add parent directory to path to import azure_openai_config
sys.path.append(os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', '..'))

load_dotenv()

from azure_openai_config import setup_azure_openai, get_azure_openai_model
setup_azure_openai()


## Summarize messages

In [2]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model=get_azure_openai_model(),
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=get_azure_openai_model(),
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)


In [3]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\nThe moon's capital is Lunapolis. The weather in Lunapolis features clear skies, with a high of 120°C and a low of -100°C. Lunapolis is home to 100,000 cheese miners, and their union is likely to strike due to dissatisfaction with the new president.", additional_kwargs={}, response_metadata={}, id='f6daca31-58ea-49c0-8feb-6202dc7e8133'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?", additional_kwargs={}, response_metadata={}, id='83aced44-9a05-42cd-b31f-fcc8d9fda5d8'),
              AIMessage(content="As the new president of Lunapolis, my response to the cheese miners' union would focus on fostering open communication, addressing their concerns, and finding a mutually beneficial resolution. Here's how I would approach the situation:\n\n1. **Acknowledge Their Concerns**: I would publicly recognize the importance of the chees

In [4]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

The moon's capital is Lunapolis. The weather in Lunapolis features clear skies, with a high of 120°C and a low of -100°C. Lunapolis is home to 100,000 cheese miners, and their union is likely to strike due to dissatisfaction with the new president.


## Trim/delete messages

In [5]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [6]:
agent = create_agent(
    model=get_azure_openai_model(),
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)


In [7]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='c9596247-a90c-4b8d-aff2-0e5f7252e41a'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='85750f71-4a91-426c-b305-0817e095cb27'),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='8e8b060c-535d-47d9-aa0f-544a09282d3d'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='ba5c3062-9edc-46fe-815b-6e5a35ad3143'),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='91671b50-d717-4f29-9bf8-a6de88de0eeb'),
              AIMessage(content="If your device feels unusually hot, it might have overheated, which can cause it to shut down or not turn on. Here's what you can do:\n\n1

In [8]:
print(response["messages"][-1].content)

If your device feels unusually hot, it might have overheated, which can cause it to shut down or not turn on. Here's what you can do:

1. **Turn it off and unplug it**: If it's still plugged in, disconnect it from the power source to let it cool down.
2. **Let it rest**: Allow the device to cool for at least 15-30 minutes in a well-ventilated area.
3. **Check for proper ventilation**: Ensure that vents or fans are not blocked by dust, debris, or objects.
4. **Try turning it on again**: Once the device has cooled down, plug it back in and attempt to power it on.

If the device is not hot, let me know, and we can troubleshoot further!
